In [1]:
import pandas as pd
df = pd.read_csv(r"path\to\your\DataCoSupplyChainDataset.csv", encoding = "latin-1")

In [2]:
print(df.shape)
print(df.columns.tolist())
df.head()

(180519, 53)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping d

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
cols = ["Order Id", "order date (DateOrders)", "Shipping Mode", "Days for shipment (scheduled)", "Days for shipping (real)", "Delivery Status",
        "Order Region", "Market", "Category Name", "Sales", "Benefit per order",
        "Order Item Discount Rate", "Customer Segment", "Type", "Product Name"]

d = df[cols].copy()

d["is_late"] = d["Days for shipping (real)"] > d["Days for shipment (scheduled)"]

print(d["Delivery Status"].value_counts())
print(d.groupby("Shipping Mode")["is_late"].mean())
print(df["Order Id"].nunique(), len(df))

Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64
Shipping Mode
First Class       1.000000
Same Day          0.478279
Second Class      0.797308
Standard Class    0.397682
Name: is_late, dtype: float64
65752 180519


In [4]:
print(d.groupby("Shipping Mode")[["Days for shipment (scheduled)", "Days for shipping (real)"]].mean())

                Days for shipment (scheduled)  Days for shipping (real)
Shipping Mode                                                          
First Class                               1.0                  2.000000
Same Day                                  0.0                  0.478279
Second Class                              2.0                  3.990828
Standard Class                            4.0                  3.995907


In [5]:
d = d[d["Delivery Status"] != "Shipping canceled"].copy()
d["native_late"] = d["Delivery Status"] == "Late delivery"
print(pd.crosstab(d["is_late"], d["native_late"]))

native_late  False  True 
is_late                  
False        73788      0
True             0  98977


In [6]:
print(d.groupby("Shipping Mode")["Days for shipping (real)"].value_counts().sort_index())

Shipping Mode   Days for shipping (real)
First Class     2                           26513
Same Day        0                            4839
                1                            4454
Second Class    2                            6819
                3                            6759
                4                            6759
                5                            6772
                6                            6697
Standard Class  2                           20873
                3                           20719
                4                           20538
                5                           20231
                6                           20792
Name: count, dtype: int64


In [7]:
d["late_if_plus1"] = d["Days for shipping (real)"] > d["Days for shipment (scheduled)"] + 1
print(d.groupby("Shipping Mode")[["is_late", "late_if_plus1"]].mean())

                 is_late  late_if_plus1
Shipping Mode                          
First Class     1.000000       0.000000
Same Day        0.479285       0.000000
Second Class    0.798290       0.598355
Standard Class  0.397691       0.201565


In [8]:
print(pd.crosstab(d["Order Region"], d["Shipping Mode"], values = d["is_late"], aggfunc = "mean").round(2))

Shipping Mode    First Class  Same Day  Second Class  Standard Class
Order Region                                                        
Canada                   1.0      0.24          0.81            0.30
Caribbean                1.0      0.52          0.82            0.37
Central Africa           1.0      0.30          0.83            0.44
Central America          1.0      0.48          0.78            0.40
Central Asia             1.0      0.20          0.90            0.43
East Africa              1.0      0.43          0.79            0.42
East of USA              1.0      0.50          0.78            0.42
Eastern Asia             1.0      0.49          0.83            0.39
Eastern Europe           1.0      0.36          0.77            0.41
North Africa             1.0      0.58          0.82            0.38
Northern Europe          1.0      0.41          0.79            0.39
Oceania                  1.0      0.43          0.79            0.41
South America            1.0      

In [9]:
sd = d[d["Shipping Mode"] == "Same Day"]
print(sd["Order Region"].value_counts().sort_values().head(8))

Order Region
Canada              29
Southern Africa     63
Central Asia        64
Central Africa      79
East Africa         95
Eastern Europe     176
North Africa       185
West Africa        212
Name: count, dtype: int64


In [10]:
print(d.groupby("is_late")["Benefit per order"].agg(["mean", "median", "count"]))

print(d.groupby(["Shipping Mode", "is_late"])["Benefit per order"].mean())

              mean  median  count
is_late                          
False    22.583197   31.68  73788
True     21.621707   31.43  98977
Shipping Mode   is_late
First Class     True       23.256392
Same Day        False      22.710397
                True       18.923781
Second Class    False      23.218506
                True       20.918756
Standard Class  False      22.503563
                True       21.320575
Name: Benefit per order, dtype: float64


In [11]:
loss = d[d["Benefit per order"] < 0]
print(len(loss), len(loss) / len(d))
print(loss["Benefit per order"].sum(), d[d["Benefit per order"] > 0]["Benefit per order"].sum())
print(loss["Category Name"].value_counts().head(10))
print(loss["is_late"].mean(), d[d["Benefit per order"] >= 0]["is_late"].mean())

32295 0.18693022313547303
-3708393.175825706 7514813.809694838
Category Name
Cleats                  4389
Men's Footwear          3993
Women's Apparel         3745
Indoor/Outdoor Games    3471
Fishing                 3047
Water Sports            2799
Camping & Hiking        2486
Cardio Equipment        2237
Shop By Sport           2061
Electronics              548
Name: count, dtype: int64
0.5744852144294782 0.572535060867089


In [12]:
multi = d[d.groupby("Order Id")["Order Id"].transform("count") > 1]
print(multi.groupby("Order Id")["Benefit per order"].nunique().mean())

3.5006489809859955


In [13]:
d["is_loss"] = d["Benefit per order"] < 0
rate = d.groupby("Category Name")["is_loss"].agg(["mean", "sum", "count"])
print(rate.sort_values("mean", ascending=False).head(10))

print(d.groupby("is_loss")["Order Item Discount Rate"].mean())

                       mean   sum  count
Category Name                           
Men's Clothing     0.242424    48    198
Health and Beauty  0.234104    81    346
Video Games        0.217662   175    804
Golf Shoes         0.217131   109    502
Pet Supplies       0.205508    97    472
DVDs               0.202614    93    459
As Seen on  TV!    0.196970    13     66
Kids' Golf Clubs   0.196629    70    356
Shop By Sport      0.196006  2061  10515
Boxing & MMA       0.194030    78    402
is_loss
False    0.101546
True     0.101967
Name: Order Item Discount Rate, dtype: float64


In [14]:
for c in ["Customer Segment", "Market", "Type"]:
    print(d.groupby(c)["is_loss"].mean().round(3), "\n")

Customer Segment
Consumer       0.186
Corporate      0.188
Home Office    0.188
Name: is_loss, dtype: float64 

Market
Africa          0.186
Europe          0.186
LATAM           0.186
Pacific Asia    0.189
USCA            0.187
Name: is_loss, dtype: float64 

Type
CASH        0.184
DEBIT       0.188
PAYMENT     0.188
TRANSFER    0.186
Name: is_loss, dtype: float64 



In [15]:
print(d["Category Name"].nunique(), d["Product Name"].nunique())

50 118


In [18]:
unit = "Product Name"

s = d.groupby(unit)["Sales"].sum().sort_values(ascending = False)
abc = pd.DataFrame({"sales": s})
abc["share"] = abc["sales"] / abc["sales"].sum()
abc["cumulative_share"] = abc["share"].cumsum()
abc["class"] = pd.cut(abc["cumulative_share"], [0, 0.8, 0.95, 1.0], labels = ["A", "B", "C"])

print(abc["class"].value_counts())
print(abc.groupby("class", observed = True)["share"].sum().round(3))
print(abc.head(10))

class
C    95
B    16
A     7
Name: count, dtype: int64
class
A    0.769
B    0.181
C    0.051
Name: share, dtype: float64
                                                      sales     share  \
Product Name                                                            
Field & Stream Sportsman 16 Gun Fire Safe      6.637668e+06  0.188493   
Perfect Fitness Perfect Rip Deck               4.233794e+06  0.120229   
Diamondback Women's Serene Classic Comfort Bi  3.946837e+06  0.112080   
Nike Men's Free 5.0+ Running Shoe              3.507549e+06  0.099605   
Nike Men's Dri-FIT Victory Golf Polo           3.011600e+06  0.085522   
Pelican Sunstream 100 Kayak                    2.967852e+06  0.084279   
O'Brien Men's Neoprene Life Vest               2.765543e+06  0.078534   
Nike Men's CJ Elite 2 TD Football Cleat        2.763977e+06  0.078490   
Under Armour Girls' Toddler Spine Surge Runni  1.214896e+06  0.034500   
Dell Laptop                                    6.375000e+05  0.018103   



In [19]:
abc["profit"] = d.groupby("Product Name")["Benefit per order"].sum()
abc["profit_share"] = abc["profit"] / abc["profit"].sum()

print(abc[["share", "profit_share", "class"]].head(10).round(3))
print(abc.groupby("class", observed=True)["profit_share"].sum().round(3))

                                               share  profit_share class
Product Name                                                            
Field & Stream Sportsman 16 Gun Fire Safe      0.188         0.192     A
Perfect Fitness Perfect Rip Deck               0.120         0.124     A
Diamondback Women's Serene Classic Comfort Bi  0.112         0.108     A
Nike Men's Free 5.0+ Running Shoe              0.100         0.095     A
Nike Men's Dri-FIT Victory Golf Polo           0.086         0.088     A
Pelican Sunstream 100 Kayak                    0.084         0.081     A
O'Brien Men's Neoprene Life Vest               0.079         0.079     A
Nike Men's CJ Elite 2 TD Football Cleat        0.078         0.078     B
Under Armour Girls' Toddler Spine Surge Runni  0.034         0.032     B
Dell Laptop                                    0.018         0.018     B
class
A    0.767
B    0.183
C    0.050
Name: profit_share, dtype: float64


In [20]:
d["is_late"] = d["is_late"].astype(int)
d["late_if_plus1"] = d["late_if_plus1"].astype(int)
d["is_loss"] = d["is_loss"].astype(int)

out_cols = ["Order Id", "order date (DateOrders)", "Shipping Mode", "Days for shipment (scheduled)", "Days for shipping (real)",
            "Order Region", "Market", "Category Name", "Product Name",
            "Sales", "Benefit per order", "is_late", "late_if_plus1", "is_loss"]
d[out_cols].to_csv("dataco_clean.csv", index = False)

abc_out = abc.reset_index()
abc_out["rank"] = range(1, len(abc_out) + 1)
abc_out.to_csv("dataco_abc.csv", index = False)